In [3]:
# ── Recargar datos desde el parquet guardado ───────────────────────────────
import pandas as pd
import os

DATA_PATH = 'C:\\Users\\farno\\OneDrive\\Desktop\\Data science & AI - Nuclio School\\proyecto final TFM\\tfm-fintech-easymoney\\data\\processed\\master_df.parquet'
df = pd.read_parquet(DATA_PATH)

# Restaurar variables necesarias
product_cols = ['short_term_deposit','loans','mortgage','funds','securities',
                'long_term_deposit','credit_card','payroll','pension_plan',
                'payroll_account','emc_account','debit_card','em_account_p',
                'em_acount']

label_map = {
    'em_acount': 'Cuenta easyMoney',
    'payroll': 'Domiciliaciones',
    'em_account_p': 'Cuenta easyMoney+',
    'debit_card': 'Tarjeta débito',
    'credit_card': 'Tarjeta crédito',
    'payroll_account': 'Cuenta nómina',
    'emc_account': 'Cuenta Crypto',
    'short_term_deposit': 'Depósito C/P',
    'long_term_deposit': 'Depósito L/P',
    'pension_plan': 'Plan pensiones',
    'funds': 'Fondos inversión',
    'securities': 'Valores',
    'mortgage': 'Hipoteca',
    'loans': 'Préstamos'
}

last_partition = df['pk_partition'].max()

print(f"✓ Datos recargados: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"✓ Última partición: {last_partition}")
print(f"✓ Productos: {len(product_cols)} columnas")

✓ Datos recargados: 5,962,924 filas × 35 columnas
✓ Última partición: 2019-05-28 00:00:00
✓ Productos: 14 columnas


In [4]:
# ── Exportar datos para Power BI ───────────────────────────────────────────

OUTPUT_PATH = '../data/processed/powerbi/'
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Tabla 1: Resumen por período
period_summary_export = df.groupby('pk_partition').agg(
    total_clients        = ('pk_cid', 'count'),
    active_clients       = ('active_customer', 'sum'),
    new_clients          = ('is_new_client', 'sum'),
    new_contracts        = ('new_contracts', 'sum'),
    avg_products         = ('total_products', 'mean'),
    clients_0_products   = ('total_products', lambda x: (x==0).sum()),
    clients_1_product    = ('total_products', lambda x: (x==1).sum()),
    clients_2plus        = ('total_products', lambda x: (x>=2).sum()),
).reset_index()
period_summary_export['pk_partition'] = period_summary_export['pk_partition'].astype(str).str[:10]
period_summary_export.to_csv(OUTPUT_PATH + 'period_summary.csv', index=False)
print(f"✓ period_summary.csv — {period_summary_export.shape}")

# Tabla 2: Penetración por producto por período
product_penetration = df.groupby('pk_partition')[product_cols].mean().mul(100).round(2).reset_index()
product_penetration['pk_partition'] = product_penetration['pk_partition'].astype(str).str[:10]
product_penetration.to_csv(OUTPUT_PATH + 'product_penetration.csv', index=False)
print(f"✓ product_penetration.csv — {product_penetration.shape}")

# Tabla 3: Perfil cliente última partición
client_profile = df[df['pk_partition'] == last_partition][[
    'pk_cid', 'segment', 'age_group', 'salary_group',
    'gender', 'region_code', 'country_id',
    'total_products', 'is_new_client',
    'client_age_months', 'active_customer'
] + product_cols].copy()
client_profile['pk_partition'] = str(last_partition)[:10]
client_profile.to_csv(OUTPUT_PATH + 'client_profile.csv', index=False)
print(f"✓ client_profile.csv — {client_profile.shape}")

# Tabla 4: KPIs resumen
kpi_summary = period_summary_export.copy()
kpi_summary['pct_new_clients'] = (kpi_summary['new_clients'] / kpi_summary['total_clients'] * 100).round(2)
kpi_summary['pct_0_products']  = (kpi_summary['clients_0_products'] / kpi_summary['total_clients'] * 100).round(2)
kpi_summary['pct_1_product']   = (kpi_summary['clients_1_product'] / kpi_summary['total_clients'] * 100).round(2)
kpi_summary['pct_crosssell']   = (kpi_summary['clients_2plus'] / kpi_summary['total_clients'] * 100).round(2)
kpi_summary.to_csv(OUTPUT_PATH + 'kpi_summary.csv', index=False)
print(f"✓ kpi_summary.csv — {kpi_summary.shape}")

print(f"\n✓ Todos los archivos exportados en: {OUTPUT_PATH}")
print("\nArchivos listos para Power BI:")
for f in os.listdir(OUTPUT_PATH):
    size = os.path.getsize(OUTPUT_PATH + f) / 1024
    print(f"  {f:<35} {size:.1f} KB")

✓ period_summary.csv — (17, 9)
✓ product_penetration.csv — (17, 15)
✓ client_profile.csv — (442995, 26)
✓ kpi_summary.csv — (17, 13)

✓ Todos los archivos exportados en: ../data/processed/powerbi/

Archivos listos para Power BI:
  client_profile.csv                  42040.3 KB
  kpi_summary.csv                     1.8 KB
  period_summary.csv                  1.4 KB
  product_penetration.csv             1.5 KB
